# **ANN**

In [7]:
pip install torch

In [8]:
"""
============================================================
  ANN (Artificial Neural Network) - Réseau de neurones simple
============================================================

"""

import numpy as np

# ============================================================
# 1) FROM SCRATCH
# ============================================================

class ANN_Scratch:
    def __init__(self, n_input, n_hidden, n_output, lr=0.1):
        # On stocke le taux d'apprentissage (learning rate)
        self.lr = lr

        # Initialisation aléatoire des poids de la couche cachée (input -> hidden)
        # Taille (n_input, n_hidden). Echelle 1.0 (au lieu de 0.1) pour que le réseau
        # sorte de la zone plate de la sigmoïde autour de 0 et apprenne le XOR correctement
        self.W1 = np.random.randn(n_input, n_hidden) * 1.0
        # Biais de la couche cachée, initialisés à zéro
        self.b1 = np.zeros((1, n_hidden))

        # Initialisation aléatoire des poids de la couche de sortie (hidden -> output)
        self.W2 = np.random.randn(n_hidden, n_output) * 1.0
        # Biais de la couche de sortie
        self.b2 = np.zeros((1, n_output))

    def sigmoid(self, x):
        # Fonction d'activation sigmoïde : écrase les valeurs entre 0 et 1
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self, x):
        # Dérivée de la sigmoïde, utilisée pendant la rétropropagation (backprop)
        return x * (1 - x)

    def forward(self, X):
        # Propagation avant : couche cachée = X . W1 + b1
        self.z1 = np.dot(X, self.W1) + self.b1
        # On applique l'activation sigmoïde sur la couche cachée
        self.a1 = self.sigmoid(self.z1)

        # Couche de sortie = a1 . W2 + b2
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        # Activation sigmoïde pour obtenir une sortie entre 0 et 1 (probabilité)
        self.a2 = self.sigmoid(self.z2)
        return self.a2

    def backward(self, X, y, output):
        m = X.shape[0]  # nombre d'exemples dans le batch

        # Erreur en sortie = différence entre prédiction et vraie valeur
        error_output = output - y
        # Gradient local de la couche de sortie (delta) = erreur * dérivée de l'activation
        delta_output = error_output * self.sigmoid_derivative(output)

        # Erreur propagée vers la couche cachée = delta_output . W2^T
        error_hidden = np.dot(delta_output, self.W2.T)
        # Gradient local de la couche cachée
        delta_hidden = error_hidden * self.sigmoid_derivative(self.a1)

        # Gradient des poids W2 = a1^T . delta_output (moyenné sur le batch)
        dW2 = np.dot(self.a1.T, delta_output) / m
        db2 = np.sum(delta_output, axis=0, keepdims=True) / m

        # Gradient des poids W1 = X^T . delta_hidden
        dW1 = np.dot(X.T, delta_hidden) / m
        db1 = np.sum(delta_hidden, axis=0, keepdims=True) / m

        # Mise à jour des poids par descente de gradient (gradient descent)
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def train(self, X, y, epochs=5000):
        for epoch in range(epochs):
            output = self.forward(X)       # propagation avant
            self.backward(X, y, output)    # rétropropagation + mise à jour
            if epoch % 1000 == 0:
                # Erreur quadratique moyenne, juste pour suivre l'entraînement
                loss = np.mean((output - y) ** 2)
                print(f"[Scratch] Epoch {epoch}, Loss = {loss:.4f}")


if __name__ == "__main__":
    # Exemple : apprendre la fonction XOR
    X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    y = np.array([[0], [1], [1], [0]])

    model = ANN_Scratch(n_input=2, n_hidden=4, n_output=1, lr=0.5)
    model.train(X, y, epochs=5000)
    print("Prédictions finales (scratch):")
    print(model.forward(X))


# ============================================================
# 2) AVEC BIBLIOTHEQUE (PyTorch)
# ============================================================
"""
Nécessite : pip install torch
"""
import torch
import torch.nn as nn
import torch.optim as optim

class ANN_Torch(nn.Module):
    def __init__(self, n_input, n_hidden, n_output):
        super().__init__()  # initialise la classe parente nn.Module
        # Couche linéaire (fully connected) : entrée -> caché
        self.fc1 = nn.Linear(n_input, n_hidden)
        # Fonction d'activation sigmoïde
        self.act = nn.Sigmoid()
        # Couche linéaire : caché -> sortie
        self.fc2 = nn.Linear(n_hidden, n_output)

    def forward(self, x):
        # On passe x dans la première couche puis l'activation
        x = self.act(self.fc1(x))
        # Puis dans la couche de sortie avec activation sigmoïde
        x = self.act(self.fc2(x))
        return x


def train_torch():
    # Données d'entraînement (XOR) en tenseurs PyTorch
    X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
    y = torch.tensor([[0.], [1.], [1.], [0.]])

    model = ANN_Torch(n_input=2, n_hidden=4, n_output=1)
    criterion = nn.MSELoss()                       # fonction de perte : erreur quadratique moyenne
    optimizer = optim.SGD(model.parameters(), lr=0.5)  # optimiseur : descente de gradient stochastique

    for epoch in range(5000):
        optimizer.zero_grad()          # on remet les gradients à zéro avant chaque passe
        output = model(X)              # propagation avant automatique
        loss = criterion(output, y)    # calcul de la perte
        loss.backward()                # rétropropagation automatique (autograd)
        optimizer.step()               # mise à jour des poids

        if epoch % 1000 == 0:
            print(f"[Torch] Epoch {epoch}, Loss = {loss.item():.4f}")

    print("Prédictions finales (torch):")
    print(model(X))


if __name__ == "__main__":
    train_torch()


[Scratch] Epoch 0, Loss = 0.2620
[Scratch] Epoch 1000, Loss = 0.2066
[Scratch] Epoch 2000, Loss = 0.1085
[Scratch] Epoch 3000, Loss = 0.0257
[Scratch] Epoch 4000, Loss = 0.0104
Prédictions finales (scratch):
[[0.07122226]
 [0.92563572]
 [0.92271623]
 [0.08551482]]
[Torch] Epoch 0, Loss = 0.2499
[Torch] Epoch 1000, Loss = 0.2168
[Torch] Epoch 2000, Loss = 0.0489
[Torch] Epoch 3000, Loss = 0.0076
[Torch] Epoch 4000, Loss = 0.0036
Prédictions finales (torch):
tensor([[0.0405],
        [0.9548],
        [0.9548],
        [0.0575]], grad_fn=<SigmoidBackward0>)


# RNN

In [9]:
"""
============================================================
  RNN (Recurrent Neural Network) - Réseau de neurones récurrent
============================================================

"""

import numpy as np

# ============================================================
# 1) FROM SCRATCH
# ============================================================

class RNN_Scratch:
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        self.hidden_size = hidden_size
        self.lr = lr

        # Poids entrée -> caché
        self.Wxh = np.random.randn(hidden_size, input_size) * 0.01
        # Poids caché -> caché (mémoire récurrente, c'est le cœur du RNN)
        self.Whh = np.random.randn(hidden_size, hidden_size) * 0.01
        # Poids caché -> sortie
        self.Why = np.random.randn(output_size, hidden_size) * 0.01
        # Biais
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))

    def forward(self, inputs):
        """
        inputs : liste de vecteurs (un par pas de temps), chacun (input_size, 1)
        Retourne la liste des sorties et des états cachés à chaque pas de temps
        """
        h = np.zeros((self.hidden_size, 1))  # état caché initial = vecteur de zéros
        self.last_inputs = inputs
        self.last_hiddens = {0: h}
        outputs = []

        for t, x in enumerate(inputs):
            # Nouvel état caché = tanh( Wxh.x + Whh.h_prev + bh )
            # C'est la formule récurrente : la mémoire dépend de l'entrée courante ET du passé
            h = np.tanh(np.dot(self.Wxh, x) + np.dot(self.Whh, h) + self.bh)
            self.last_hiddens[t + 1] = h
            # Sortie au pas de temps t = Why.h + by
            y = np.dot(self.Why, h) + self.by
            outputs.append(y)

        return outputs, h

    def backward(self, d_outputs):
        """Rétropropagation à travers le temps (Backpropagation Through Time - BPTT)"""
        n = len(self.last_inputs)
        dWxh = np.zeros_like(self.Wxh)
        dWhh = np.zeros_like(self.Whh)
        dWhy = np.zeros_like(self.Why)
        dbh = np.zeros_like(self.bh)
        dby = np.zeros_like(self.by)
        dh_next = np.zeros((self.hidden_size, 1))  # gradient venant du futur (t+1)

        # On parcourt le temps à l'envers (du dernier pas au premier) : c'est le "through time"
        for t in reversed(range(n)):
            dy = d_outputs[t]  # gradient de la perte par rapport à la sortie au temps t
            dWhy += np.dot(dy, self.last_hiddens[t + 1].T)
            dby += dy

            # gradient qui arrive sur l'état caché : contribution directe + celle du futur
            dh = np.dot(self.Why.T, dy) + dh_next
            # dérivée de tanh : (1 - h^2)
            dh_raw = (1 - self.last_hiddens[t + 1] ** 2) * dh

            dbh += dh_raw
            dWxh += np.dot(dh_raw, self.last_inputs[t].T)
            dWhh += np.dot(dh_raw, self.last_hiddens[t].T)

            # gradient à propager vers le pas de temps précédent
            dh_next = np.dot(self.Whh.T, dh_raw)

        # Clipping des gradients pour éviter l'explosion (problème classique des RNN)
        for grad in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(grad, -5, 5, out=grad)

        # Mise à jour des poids par descente de gradient
        self.Wxh -= self.lr * dWxh
        self.Whh -= self.lr * dWhh
        self.Why -= self.lr * dWhy
        self.bh -= self.lr * dbh
        self.by -= self.lr * dby


if __name__ == "__main__":
    # Exemple : séquence de 3 vecteurs de taille 4, on veut prédire une sortie de taille 2
    inputs = [np.random.randn(4, 1) for _ in range(3)]
    targets = [np.random.randn(2, 1) for _ in range(3)]

    rnn = RNN_Scratch(input_size=4, hidden_size=5, output_size=2, lr=0.05)

    for epoch in range(200):
        outputs, h = rnn.forward(inputs)
        # Gradient de la MSE par rapport à chaque sortie : 2*(y_pred - y_true)
        d_outputs = [2 * (outputs[t] - targets[t]) for t in range(len(inputs))]
        rnn.backward(d_outputs)
        if epoch % 50 == 0:
            loss = np.mean([(outputs[t] - targets[t]) ** 2 for t in range(len(inputs))])
            print(f"[Scratch RNN] Epoch {epoch}, Loss = {loss:.4f}")


# ============================================================
# 2) AVEC BIBLIOTHEQUE (PyTorch)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim

class RNN_Torch(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # nn.RNN gère automatiquement la boucle temporelle et la mémoire cachée
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        # Couche linéaire pour transformer l'état caché final en sortie
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x : (batch, seq_len, input_size)
        out, h_n = self.rnn(x)  # out : sorties à chaque pas de temps, h_n : dernier état caché
        # On prend la sortie du dernier pas de temps pour la prédiction
        out = self.fc(out[:, -1, :])
        return out


def train_torch_demo():
    # Batch de 2 séquences, longueur 5, dimension d'entrée 3
    X = torch.randn(2, 5, 3)
    y = torch.randn(2, 1)

    model = RNN_Torch(input_size=3, hidden_size=8, output_size=1)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(200):
        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        loss.backward()  # PyTorch fait automatiquement le BPTT (backprop through time)
        optimizer.step()
        if epoch % 50 == 0:
            print(f"[Torch RNN] Epoch {epoch}, Loss = {loss.item():.4f}")


if __name__ == "__main__":
    train_torch_demo()


[Scratch RNN] Epoch 0, Loss = 0.4852
[Scratch RNN] Epoch 50, Loss = 0.0001
[Scratch RNN] Epoch 100, Loss = 0.0000
[Scratch RNN] Epoch 150, Loss = 0.0000
[Torch RNN] Epoch 0, Loss = 3.1474
[Torch RNN] Epoch 50, Loss = 0.0050
[Torch RNN] Epoch 100, Loss = 0.0000
[Torch RNN] Epoch 150, Loss = 0.0000


# LSTM

In [10]:
"""
============================================================
  LSTM (Long Short-Term Memory)
============================================================
"""

import numpy as np


def sigmoid(x):
    # Fonction d'activation sigmoïde, utilisée pour les portes (valeurs entre 0 et 1)
    return 1 / (1 + np.exp(-x))


# ============================================================
# 1) FROM SCRATCH (NumPy)
# ============================================================

class LSTM_Scratch:
    def __init__(self, input_size, hidden_size, lr=0.05):
        self.hidden_size = hidden_size
        self.lr = lr
        z_size = input_size + hidden_size  # on concatène x et h pour simplifier les calculs

        # Chaque porte a ses propres poids (W) et biais (b), appliqués sur [h_prev, x]
        # Porte d'oubli (forget gate) : décide quoi effacer de la mémoire
        self.Wf = np.random.randn(hidden_size, z_size) * 0.01
        self.bf = np.zeros((hidden_size, 1))

        # Porte d'entrée (input gate) : décide quoi ajouter à la mémoire
        self.Wi = np.random.randn(hidden_size, z_size) * 0.01
        self.bi = np.zeros((hidden_size, 1))

        # Candidat de cellule (valeurs candidates à ajouter à la mémoire)
        self.Wc = np.random.randn(hidden_size, z_size) * 0.01
        self.bc = np.zeros((hidden_size, 1))

        # Porte de sortie (output gate) : décide quoi sortir de la mémoire
        self.Wo = np.random.randn(hidden_size, z_size) * 0.01
        self.bo = np.zeros((hidden_size, 1))

    def step(self, x, h_prev, c_prev):
        """Un seul pas de temps du LSTM"""
        # Concaténation de l'état caché précédent et de l'entrée courante
        z = np.vstack((h_prev, x))

        # Porte d'oubli : quelle proportion de c_prev on garde (entre 0 et 1)
        f = sigmoid(np.dot(self.Wf, z) + self.bf)

        # Porte d'entrée : quelle proportion du nouveau candidat on ajoute
        i = sigmoid(np.dot(self.Wi, z) + self.bi)

        # Candidat de nouvelles valeurs de mémoire (entre -1 et 1 grâce à tanh)
        c_candidate = np.tanh(np.dot(self.Wc, z) + self.bc)

        # Nouvelle cellule mémoire = ce qu'on garde de l'ancien + ce qu'on ajoute de nouveau
        c = f * c_prev + i * c_candidate

        # Porte de sortie : quelle partie de la mémoire on expose comme état caché
        o = sigmoid(np.dot(self.Wo, z) + self.bo)

        # Nouvel état caché = porte de sortie * tanh(cellule mémoire)
        h = o * np.tanh(c)

        return h, c

    def forward(self, inputs):
        """inputs : liste de vecteurs colonnes (input_size, 1)"""
        h = np.zeros((self.hidden_size, 1))
        c = np.zeros((self.hidden_size, 1))
        hidden_states = []
        for x in inputs:
            h, c = self.step(x, h, c)
            hidden_states.append(h)
        return hidden_states, h, c


if __name__ == "__main__":
    # Séquence d'exemple : 4 pas de temps, dimension d'entrée 3
    inputs = [np.random.randn(3, 1) for _ in range(4)]
    lstm = LSTM_Scratch(input_size=3, hidden_size=5)
    hidden_states, h_final, c_final = lstm.forward(inputs)

    print("Etats cachés à chaque pas de temps (scratch LSTM):")
    for t, h in enumerate(hidden_states):
        print(f"t={t}: {h.ravel()}")
    print("\nNote : l'entraînement complet (backprop through time) suit la même")
    print("logique que pour le RNN, mais avec un gradient qui passe aussi par c_t.")


# ============================================================
# 2) AVEC BIBLIOTHEQUE (PyTorch)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim

class LSTM_Torch(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # nn.LSTM implémente automatiquement les 4 portes et la cell state
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        # Couche de sortie
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # out : sortie à chaque pas de temps ; (h_n, c_n) : derniers états caché et cellule
        out, (h_n, c_n) = self.lstm(x)
        # On utilise la sortie du dernier pas de temps pour la prédiction finale
        out = self.fc(out[:, -1, :])
        return out


def train_torch_demo():
    X = torch.randn(2, 6, 3)  # batch=2, seq_len=6, input_size=3
    y = torch.randn(2, 1)

    model = LSTM_Torch(input_size=3, hidden_size=8, output_size=1)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(200):
        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        loss.backward()  # gère automatiquement le gradient à travers les portes et le temps
        optimizer.step()
        if epoch % 50 == 0:
            print(f"[Torch LSTM] Epoch {epoch}, Loss = {loss.item():.4f}")


if __name__ == "__main__":
    train_torch_demo()


Etats cachés à chaque pas de temps (scratch LSTM):
t=0: [ 0.00185662 -0.00184879  0.00236821  0.00039691  0.0009051 ]
t=1: [ 0.00396409 -0.00118547  0.0035303   0.00037963 -0.00029206]
t=2: [ 0.00080516 -0.003102    0.0022903   0.00114262  0.00258997]
t=3: [ 1.01924410e-02 -3.47778173e-05  6.94233704e-03  4.43530581e-03
 -1.18560691e-03]

Note : l'entraînement complet (backprop through time) suit la même
logique que pour le RNN, mais avec un gradient qui passe aussi par c_t.
[Torch LSTM] Epoch 0, Loss = 0.2595
[Torch LSTM] Epoch 50, Loss = 0.0001
[Torch LSTM] Epoch 100, Loss = 0.0000
[Torch LSTM] Epoch 150, Loss = 0.0000


# GRU

In [11]:
"""
============================================================
  GRU (Gated Recurrent Unit)
============================================================
"""

import numpy as np


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


# ============================================================
# 1) FROM SCRATCH
# ============================================================

class GRU_Scratch:
    def __init__(self, input_size, hidden_size):
        z_size = input_size + hidden_size

        # Porte de mise à jour (update gate) : décide combien du passé on garde
        self.Wz = np.random.randn(hidden_size, z_size) * 0.01
        self.bz = np.zeros((hidden_size, 1))

        # Porte de réinitialisation (reset gate) : décide combien du passé on "oublie"
        # pour calculer le nouveau candidat
        self.Wr = np.random.randn(hidden_size, z_size) * 0.01
        self.br = np.zeros((hidden_size, 1))

        # Candidat pour le nouvel état caché
        self.Wh = np.random.randn(hidden_size, z_size) * 0.01
        self.bh = np.zeros((hidden_size, 1))

        self.hidden_size = hidden_size

    def step(self, x, h_prev):
        # Concaténation entrée + état caché précédent
        z_input = np.vstack((h_prev, x))

        # Porte de mise à jour : entre 0 (garde tout l'ancien) et 1 (prend tout le nouveau)
        z = sigmoid(np.dot(self.Wz, z_input) + self.bz)

        # Porte de réinitialisation : contrôle l'influence de h_prev sur le candidat
        r = sigmoid(np.dot(self.Wr, z_input) + self.br)

        # Pour le candidat, on applique la porte reset SUR h_prev avant de concaténer
        reset_input = np.vstack((r * h_prev, x))
        h_candidate = np.tanh(np.dot(self.Wh, reset_input) + self.bh)

        # Etat caché final = interpolation entre l'ancien état et le candidat,
        # pondérée par la porte de mise à jour z
        h = (1 - z) * h_prev + z * h_candidate

        return h

    def forward(self, inputs):
        h = np.zeros((self.hidden_size, 1))
        hidden_states = []
        for x in inputs:
            h = self.step(x, h)
            hidden_states.append(h)
        return hidden_states, h


if __name__ == "__main__":
    inputs = [np.random.randn(3, 1) for _ in range(4)]
    gru = GRU_Scratch(input_size=3, hidden_size=5)
    hidden_states, h_final = gru.forward(inputs)

    print("Etats cachés à chaque pas de temps (scratch GRU):")
    for t, h in enumerate(hidden_states):
        print(f"t={t}: {h.ravel()}")


# ============================================================
# 2) AVEC BIBLIOTHEQUE (PyTorch)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim

class GRU_Torch(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # nn.GRU implémente automatiquement les portes reset/update
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, h_n = self.gru(x)         # out : séquence de sorties, h_n : dernier état caché
        out = self.fc(out[:, -1, :])   # on utilise le dernier pas de temps pour prédire
        return out


def train_torch_demo():
    X = torch.randn(2, 6, 3)
    y = torch.randn(2, 1)

    model = GRU_Torch(input_size=3, hidden_size=8, output_size=1)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(200):
        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f"[Torch GRU] Epoch {epoch}, Loss = {loss.item():.4f}")


if __name__ == "__main__":
    train_torch_demo()


Etats cachés à chaque pas de temps (scratch GRU):
t=0: [-0.01081283  0.00221361 -0.00165296 -0.00416372 -0.0077603 ]
t=1: [-0.0261938   0.00248572 -0.00283548 -0.00623671 -0.01426273]
t=2: [-0.00611894  0.0018252  -0.00200176 -0.00248861 -0.00565784]
t=3: [ 0.00118615  0.00216219 -0.00331065 -0.0003885  -0.00365605]
[Torch GRU] Epoch 0, Loss = 1.2937
[Torch GRU] Epoch 50, Loss = 0.0021
[Torch GRU] Epoch 100, Loss = 0.0000
[Torch GRU] Epoch 150, Loss = 0.0000


# SELF ATTENTION

In [12]:
"""
============================================================
  Self-Attention
============================================================

"""

import numpy as np


def softmax(x, axis=-1):
    # On soustrait le max pour la stabilité numérique (évite les overflow d'exponentielle)
    x = x - np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)


# ============================================================
# 1) FROM SCRATCH
# ============================================================

class SelfAttention_Scratch:
    def __init__(self, d_model, d_k):
        # d_model : dimension des embeddings d'entrée
        # d_k : dimension des vecteurs Q, K, V (souvent d_k = d_model)
        self.d_k = d_k

        # Matrices de projection apprises, qui transforment X en Q, K, V
        self.Wq = np.random.randn(d_model, d_k) * 0.01
        self.Wk = np.random.randn(d_model, d_k) * 0.01
        self.Wv = np.random.randn(d_model, d_k) * 0.01

    def forward(self, X):
        """
        X : (seq_len, d_model) - la séquence d'embeddings d'entrée
        """
        # Projection de X en Query, Key, Value
        Q = np.dot(X, self.Wq)   # (seq_len, d_k)
        K = np.dot(X, self.Wk)   # (seq_len, d_k)
        V = np.dot(X, self.Wv)   # (seq_len, d_k)

        # Scores d'attention brut = similarité entre chaque paire de mots (produit scalaire Q.K^T)
        scores = np.dot(Q, K.T)  # (seq_len, seq_len)

        # Mise à l'échelle par sqrt(d_k) pour stabiliser les gradients
        # (évite que le softmax devienne trop "piqué" quand d_k est grand)
        scores = scores / np.sqrt(self.d_k)

        # Normalisation en probabilités : chaque ligne somme à 1
        # attn_weights[i][j] = combien le mot i doit "prêter attention" au mot j
        attn_weights = softmax(scores, axis=-1)

        # Sortie = moyenne pondérée des Values, pondérée par les poids d'attention
        output = np.dot(attn_weights, V)  # (seq_len, d_k)

        return output, attn_weights


if __name__ == "__main__":
    # Séquence de 4 mots (tokens), chacun représenté par un embedding de taille 8
    seq_len, d_model, d_k = 4, 8, 8
    X = np.random.randn(seq_len, d_model)

    attn = SelfAttention_Scratch(d_model=d_model, d_k=d_k)
    output, weights = attn.forward(X)

    print("Poids d'attention (chaque ligne somme à 1) :")
    print(np.round(weights, 3))
    print("\nSortie de self-attention (contexte enrichi pour chaque mot) :")
    print(output)


# ============================================================
# 2) AVEC BIBLIOTHEQUE (PyTorch)
# ============================================================
import torch
import torch.nn as nn

class SelfAttention_Torch(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        # nn.MultiheadAttention implémente Q/K/V, le scaling et le softmax automatiquement
        # n_heads : nombre de têtes d'attention (multi-head attention)
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)

    def forward(self, x):
        # En self-attention, Query, Key et Value viennent tous de la même entrée x
        output, attn_weights = self.attn(x, x, x)
        return output, attn_weights


def demo_torch():
    # batch=1, seq_len=4, d_model=8
    x = torch.randn(1, 4, 8)
    model = SelfAttention_Torch(d_model=8, n_heads=2)  # 2 têtes d'attention

    output, weights = model(x)
    print("[Torch] Sortie de self-attention:", output.shape)
    print("[Torch] Poids d'attention (moyennés sur les têtes):", weights.shape)


if __name__ == "__main__":
    demo_torch()


Poids d'attention (chaque ligne somme à 1) :
[[0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]]

Sortie de self-attention (contexte enrichi pour chaque mot) :
[[ 0.02216734  0.01179817  0.01989825  0.00710382  0.00745032 -0.00451897
  -0.01298145 -0.01063967]
 [ 0.02217029  0.01179152  0.01989487  0.00709713  0.00743792 -0.0045115
  -0.01297637 -0.01063646]
 [ 0.02217132  0.01179858  0.01990082  0.00710096  0.00744895 -0.00451739
  -0.01298286 -0.01063896]
 [ 0.02216799  0.01179817  0.01989565  0.00710424  0.00744921 -0.0045173
  -0.01298297 -0.01064134]]
[Torch] Sortie de self-attention: torch.Size([1, 4, 8])
[Torch] Poids d'attention (moyennés sur les têtes): torch.Size([1, 4, 4])


# TRANSFORMER

In [13]:
"""
============================================================
  Transformer (bloc encodeur simplifié)
============================================================
"""

import numpy as np


def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)


def layer_norm(x, eps=1e-6):
    # Normalisation sur la dernière dimension : centre-réduit chaque vecteur (moyenne 0, variance 1)
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)


def relu(x):
    return np.maximum(0, x)


# ============================================================
# 1) FROM SCRATCH
# ============================================================

class TransformerEncoderBlock_Scratch:
    def __init__(self, d_model, d_ff):
        # --- Paramètres de la self-attention (une seule tête, pour simplifier) ---
        self.Wq = np.random.randn(d_model, d_model) * 0.01
        self.Wk = np.random.randn(d_model, d_model) * 0.01
        self.Wv = np.random.randn(d_model, d_model) * 0.01
        self.Wo = np.random.randn(d_model, d_model) * 0.01  # projection de sortie de l'attention

        # --- Paramètres du réseau feed-forward (2 couches denses) ---
        self.W1 = np.random.randn(d_model, d_ff) * 0.01
        self.b1 = np.zeros((1, d_ff))
        self.W2 = np.random.randn(d_ff, d_model) * 0.01
        self.b2 = np.zeros((1, d_model))

        self.d_model = d_model

    def self_attention(self, X):
        Q = np.dot(X, self.Wq)
        K = np.dot(X, self.Wk)
        V = np.dot(X, self.Wv)

        # Scores d'attention mis à l'échelle
        scores = np.dot(Q, K.T) / np.sqrt(self.d_model)
        weights = softmax(scores, axis=-1)

        # Sortie pondérée par l'attention, puis projection finale Wo
        attn_output = np.dot(weights, V)
        return np.dot(attn_output, self.Wo)

    def feed_forward(self, X):
        # Première couche dense + ReLU
        hidden = relu(np.dot(X, self.W1) + self.b1)
        # Deuxième couche dense, retour à la dimension d_model
        return np.dot(hidden, self.W2) + self.b2

    def forward(self, X):
        """
        X : (seq_len, d_model)
        """
        # --- Sous-couche 1 : Self-Attention + connexion résiduelle + normalisation ---
        attn_out = self.self_attention(X)
        X = layer_norm(X + attn_out)  # "Add & Norm" : on additionne l'entrée d'origine (résiduel)

        # --- Sous-couche 2 : Feed-Forward + connexion résiduelle + normalisation ---
        ff_out = self.feed_forward(X)
        X = layer_norm(X + ff_out)    # deuxième "Add & Norm"

        return X


if __name__ == "__main__":
    seq_len, d_model, d_ff = 4, 8, 16
    X = np.random.randn(seq_len, d_model)  # séquence d'embeddings d'entrée

    block = TransformerEncoderBlock_Scratch(d_model=d_model, d_ff=d_ff)
    output = block.forward(X)

    print("Sortie du bloc encodeur Transformer (scratch):")
    print(output)
    print("\nForme de sortie (doit être identique à l'entrée):", output.shape)


# ============================================================
# 2) AVEC BIBLIOTHEQUE (PyTorch)
# ============================================================
import torch
import torch.nn as nn

class Transformer_Torch(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, n_layers):
        super().__init__()
        # Un TransformerEncoderLayer = self-attention multi-têtes + feed-forward + Add&Norm
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,      # dimension des embeddings
            nhead=n_heads,        # nombre de têtes d'attention
            dim_feedforward=d_ff, # taille de la couche cachée du feed-forward
            batch_first=True
        )
        # On empile n_layers blocs encodeurs identiques
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, x):
        # x : (batch, seq_len, d_model)
        return self.encoder(x)


def demo_torch():
    x = torch.randn(1, 5, 16)  # batch=1, seq_len=5, d_model=16
    model = Transformer_Torch(d_model=16, n_heads=4, d_ff=32, n_layers=2)
    output = model(x)
    print("[Torch] Sortie du Transformer:", output.shape)


if __name__ == "__main__":
    demo_torch()


Sortie du bloc encodeur Transformer (scratch):
[[ 1.7416684  -0.50394573 -1.09456557 -0.77715452  1.36975736 -0.89003732
  -0.26218163  0.41645901]
 [-0.41055093 -0.3341633  -0.68963895 -0.22713618 -1.65170781  0.44506533
   1.15723551  1.71089633]
 [ 1.47917848  1.43330547 -0.52364889  0.58112594 -1.27162761 -0.7323062
  -0.99576575  0.02973856]
 [-0.51258498  1.840691   -1.61178832  0.71096276  0.64692888  0.18416794
  -0.65733261 -0.60104466]]

Forme de sortie (doit être identique à l'entrée): (4, 8)
[Torch] Sortie du Transformer: torch.Size([1, 5, 16])
